# 01 — Data cleaning

**Does:** cleans 5 years of daily state-level demand data (Jan 2020 – Mar 2025).

**Input:** `data/raw/india_electricity_data.csv`

**Output:** `data/processed/india_electricity_data_updated.csv`

**Steps:** fix column-shifted rows and impossible values → drop Essar Steel and Jharkhand → merge DD + DNH → one-day spikes to missing → forward/backward fill nulls within each state → checks.

In [1]:
import pandas as pd


In [2]:
df=pd.read_csv("../data/raw/india_electricity_data.csv")


In [3]:
df.head()

,Date,State,Max Demand Met,Shortage During Peak,Energy Met,Drawl Schedule,OD(+) / UD(-),Max OD,Energy Shortage
0,2020-01-01 00:00:00,Andhra Pradesh,8561.0,0.0,170.1,66.3,-0.5,367.0,0.0
1,2020-01-01 00:00:00,Arunachal Pradesh,109.0,1.0,2.2,1.9,0.2,34.0,0.0
2,2020-01-01 00:00:00,Assam,1358.0,28.0,23.4,17.3,1.3,90.0,0.2
3,2020-01-01 00:00:00,Bihar,4656.0,0.0,82.0,81.2,-0.7,255.0,0.0
4,2020-01-01 00:00:00,Chandigarh,306.0,0.0,4.9,4.7,0.2,46.0,0.0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 69012 entries, 0 to 69011
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Date                  69012 non-null  str    
 1   State                 69012 non-null  str    
 2   Max Demand Met        62972 non-null  float64
 3   Shortage During Peak  64006 non-null  float64
 4   Energy Met            64014 non-null  float64
 5   Drawl Schedule        64024 non-null  float64
 6   OD(+) / UD(-)         64024 non-null  float64
 7   Max OD                64020 non-null  float64
 8   Energy Shortage       64024 non-null  float64
dtypes: float64(7), str(2)
memory usage: 4.7 MB


In [5]:
df.isnull().sum() # is find how many null values are present in the dataset

Date                       0
State                      0
Max Demand Met          6040
Shortage During Peak    5006
Energy Met              4998
Drawl Schedule          4988
OD(+) / UD(-)           4988
Max OD                  4992
Energy Shortage         4988
dtype: int64

Checking the total missing values for each state.

In [6]:
df[['Max Demand Met', 'Energy Met', 'Shortage During Peak', 'OD(+) / UD(-)', 'Energy Shortage', 'Max OD']].isnull().groupby(df['State']).sum()

,Max Demand Met,Energy Met,Shortage During Peak,OD(+) / UD(-),Energy Shortage,Max OD
State,,,,,,
Andhra Pradesh,43,43,43,43,43,43
Arunachal Pradesh,43,43,43,43,43,43
Assam,29,29,29,29,29,29
Bihar,29,29,29,29,29,29
Chandigarh,29,29,29,29,29,29
Chhattisgarh,30,30,30,30,30,30
DD,1056,1056,1056,1053,1053,1053
DNH,1056,29,29,29,29,29
DVC,29,29,29,29,29,29


## Fix column-shifted rows (DNH) and corrupted rows (HP)

**Detected by:** `Max Demand Met` is NaN but `Shortage During Peak` is not. 1,034 raw rows match. They are two different problems:

**1. DNH, 1,027 rows (June 2022 onward): shifted one column right.** From June 2022 DD stops reporting separately and DNH carries the merged UT total, but every value sits one column too far right: the real peak demand (~1,250 MW) is under `Shortage During Peak`, the real energy (~29 MU) under `Drawl Schedule`, and so on. **Fix:** shift those rows one column left. The last column (`Energy Shortage`) is lost off the end and set to 0, because DNH's shortage is 0 on 96% of days.

**2. HP, 7 rows: corrupted, not shifted.** Read either way, the values don't fit HP (shortage 2.0, energy 0, drawl 218–651 when HP's daily energy is ~35 MU, negative energy shortage). They fall on the same dates DD and DNH are missing, so they look like a scraping error on those days. **Fix:** set all their numbers to NaN; the ffill below fills them from the previous day.

**Why it matters:** before this fix the ffill copied a stale value (1,237 MW) over ~1,000 days of DD & DNH demand, `Energy Met` was 0, and the real demand was counted as **shortage**: 62% of the national shortage figure was this bug (1,085 MW vs a real 412 MW on average).

Found in `02_merit_order.ipynb`; logged in `docs/model_log.md`.

In [7]:
import numpy as np

numeric_cols = ["Max Demand Met", "Shortage During Peak", "Energy Met",
                "Drawl Schedule", "OD(+) / UD(-)", "Max OD", "Energy Shortage"]

flagged = df["Max Demand Met"].isna() & df["Shortage During Peak"].notna()
print("Flagged rows:", flagged.sum())
print(df.loc[flagged, "State"].value_counts())

# 1. DNH: shift one column left
shifted = flagged & (df["State"] == "DNH")
df.loc[shifted, numeric_cols] = df.loc[shifted, numeric_cols].shift(-1, axis=1)
df.loc[shifted, "Energy Shortage"] = 0

# 2. HP: corrupted rows -> missing
corrupted = flagged & (df["State"] == "HP")
df.loc[corrupted, numeric_cols] = np.nan

print(f"Shifted back: {shifted.sum()} | set to missing: {corrupted.sum()}")
print("DNH after fix:", df.loc[shifted, "Max Demand Met"].describe()[["min", "mean", "max"]].round(0).to_dict())

Flagged rows: 1034
State
DNH    1027
HP        7
Name: count, dtype: int64
Shifted back: 1027 | set to missing: 7
DNH after fix: {'min': 346.0, 'mean': 1247.0, 'max': 1390.0}


## Impossible values → missing

A few days report values that are physically impossible: these are reporting errors or unreported days.

| Rule | Rows | Example |
|---|---|---|
| `Energy Met = 0` | 10 | HP, MP, Goa, DNH on single days |
| `Energy Shortage < 0` | 16 | HP and MP, mostly the same days |
| `Shortage During Peak < 0` | 1 | Kerala, 29 Jan 2020: −75 MW, with 0 on the days either side |

They are set to NaN so the ffill below fills them from the previous day, instead of keeping a wrong number.

In [8]:
print("Energy Met == 0:", (df["Energy Met"] == 0).sum(),
      "| Energy Shortage < 0:", (df["Energy Shortage"] < 0).sum(),
      "| Shortage During Peak < 0:", (df["Shortage During Peak"] < 0).sum())

df.loc[df["Energy Met"] == 0, "Energy Met"] = np.nan
df.loc[df["Energy Shortage"] < 0, "Energy Shortage"] = np.nan
df.loc[df["Shortage During Peak"] < 0, "Shortage During Peak"] = np.nan

Energy Met == 0: 10 | Energy Shortage < 0: 16 | Shortage During Peak < 0: 1


In [9]:
# Check: nothing shifted or impossible remains
assert len(set(df.loc[flagged, 'State']) - {'DNH', 'HP'}) == 0
assert (df["Max Demand Met"].isna() & df["Shortage During Peak"].notna()).sum() == 0
assert (df["Energy Met"] == 0).sum() == 0
assert (df["Energy Shortage"] < 0).sum() == 0
assert (df["Shortage During Peak"] < 0).sum() == 0
print("Raw data fixes OK")

Raw data fixes OK


In [10]:
df_updated = df[~df['State'].isin(['Essar steel', 'Jharkhand'])].copy()

Exclusions Jharkhand, Essar.

In [11]:
df_updated[['Max Demand Met', 'Energy Met', 'Shortage During Peak', 'OD(+) / UD(-)', 'Energy Shortage', 'Max OD']].isnull().groupby(df_updated['State']).sum()

,Max Demand Met,Energy Met,Shortage During Peak,OD(+) / UD(-),Energy Shortage,Max OD
State,,,,,,
Andhra Pradesh,43,43,43,43,43,43
Arunachal Pradesh,43,43,43,43,43,43
Assam,29,29,29,29,29,29
Bihar,29,29,29,29,29,29
Chandigarh,29,29,29,29,29,29
Chhattisgarh,30,30,30,30,30,30
DD,1056,1056,1056,1053,1056,1053
DNH,29,31,29,29,29,29
DVC,29,29,29,29,29,29


Adding DD and DNH

In [12]:
dd_dnh_states = ['DD', 'DNH']

dd_dnh_daily = (
    df_updated[df_updated['State'].isin(dd_dnh_states)]
    .groupby('Date', as_index=False)
    .sum(numeric_only=True, min_count=1)
)
dd_dnh_daily['State'] = 'DD & DNH'

dd_dnh_daily = dd_dnh_daily[
    ['Date', 'State']
    + [col for col in df_updated.columns if col not in ['Date', 'State']]
]

df_updated = pd.concat(
    [df_updated[~df_updated['State'].isin(dd_dnh_states)], dd_dnh_daily],
    ignore_index=True,
).sort_values(['Date', 'State'], ignore_index=True)

In [13]:
df_updated[['Max Demand Met', 'Energy Met', 'Shortage During Peak', 'OD(+) / UD(-)', 'Energy Shortage', 'Max OD']].isnull().groupby(df_updated['State']).sum()

,Max Demand Met,Energy Met,Shortage During Peak,OD(+) / UD(-),Energy Shortage,Max OD
State,,,,,,
Andhra Pradesh,43,43,43,43,43,43
Arunachal Pradesh,43,43,43,43,43,43
Assam,29,29,29,29,29,29
Bihar,29,29,29,29,29,29
Chandigarh,29,29,29,29,29,29
Chhattisgarh,30,30,30,30,30,30
DD & DNH,29,29,29,26,29,26
DVC,29,29,29,29,29,29
Delhi,29,29,29,29,29,30


## One-day spikes → missing (added in Part 2, model log Step 16a)

Found while building forecasting features: a few days have `Max Demand Met` values that are obviously typos, e.g. Telangana 236 MW on 1 Oct 2023 between days of ~12,400 MW, Tamil Nadu 933 MW on 10 Nov 2021, Sikkim 1,700 MW on 10 Mar 2021 between days of ~90 MW, HP 4,309 MW on 2 Sep 2024 (normally ~1,700).

**Rule:** a day whose `Max Demand Met` is more than 65% away from the median of its own 7-day window (|log(value / median)| > 0.5) is set to NaN, so the fill below replaces it from the previous day. The median is used because one bad day can't move it. Real swings in large states are far smaller (the COVID lockdown drop of 25 Mar 2020 *stays* low, so it never stands out from its week).

**Except real events:** four flagged dips fall on cyclone landfalls, when demand really did collapse for a day; they are kept: Goa 17 May 2021 (Tauktae), Bihar 28 May 2021 (Yaas), Mizoram 28 May 2024 (Remal), Puducherry 1 Dec 2024 (Fengal). The other flagged days are treated as errors; a few small-state dips (Sikkim, Tripura, Chandigarh, Arunachal) and Rajasthan 4 Feb 2024 (7,976 MW vs ~17,000) could be real local outages, but none matches a known event.

Only `Max Demand Met` is cleaned this way: it is the column every model uses. `Energy Met` has similar-looking dips, but several are real cyclone days and it isn't used.

In [14]:
SPIKE_LIMIT = 0.5   # |log(value / 7-day median)|: 0.5 means more than ~65% above or ~40% below
REAL_EVENTS = {     # flagged dips that are real: cyclone landfalls
    ("Goa", "2021-05-17"): "Cyclone Tauktae",
    ("Bihar", "2021-05-28"): "Cyclone Yaas",
    ("Mizoram", "2024-05-28"): "Cyclone Remal",
    ("Puducherry", "2024-12-01"): "Cyclone Fengal",
}

df_updated = df_updated.sort_values(['State', 'Date']).reset_index(drop=True)
col = 'Max Demand Met'
median = df_updated.groupby('State')[col].transform(lambda s: s.rolling(7, center=True, min_periods=4).median())
spike = np.log(df_updated[col] / median).abs() > SPIKE_LIMIT
real = pd.Series([(s, str(d)[:10]) in REAL_EVENTS for s, d in zip(df_updated['State'], df_updated['Date'])],
                 index=df_updated.index)
assert (spike & real).sum() == len(REAL_EVENTS), "a listed real event is no longer flagged: check the list"

fixed = spike & ~real
spikes = df_updated.loc[spike, ['Date', 'State', col]].assign(
    Median=median[spike].round(0),
    Action=np.where(real[spike], "kept (" + pd.Series([REAL_EVENTS.get((s, str(d)[:10]), "") for s, d in
                                         zip(df_updated.loc[spike, 'State'], df_updated.loc[spike, 'Date'])],
                                         index=df_updated.index[spike]) + ")", "set to missing"))
df_updated.loc[fixed, col] = np.nan
print(f"{spike.sum()} flagged: {fixed.sum()} set to missing, {(spike & real).sum()} kept as real events")
spikes.sort_values(['State', 'Date'])

25 flagged: 21 set to missing, 4 kept as real events


,Date,State,Max Demand Met,Median,Action
2042,2020-05-05 00:00:00,Arunachal Pradesh,60.0,102.0,set to missing
6264,2021-05-28 00:00:00,Bihar,3087.0,5529.0,kept (Cyclone Yaas)
8451,2022-02-22 00:00:00,Chandigarh,103.0,206.0,set to missing
12498,2022-09-23 00:00:00,DD & DNH,362.0,1206.0,set to missing
12836,2023-08-27 00:00:00,DD & DNH,346.0,1299.0,set to missing
13250,2024-10-14 00:00:00,DD & DNH,396.0,1311.0,set to missing
19395,2020-08-13 00:00:00,Goa,64.0,363.0,set to missing
19396,2020-08-14 00:00:00,Goa,64.0,352.0,set to missing
19672,2021-05-17 00:00:00,Goa,257.0,467.0,kept (Cyclone Tauktae)
24710,2024-09-02 00:00:00,HP,4309.0,1703.0,set to missing


In [15]:
numeric_cols = ['Max Demand Met', 'Shortage During Peak', 'Energy Met', 
                 'Drawl Schedule', 'OD(+) / UD(-)', 'Max OD', 'Energy Shortage']

df_updated = df_updated.sort_values(['State', 'Date']).reset_index(drop=True)
df_updated[numeric_cols] = df_updated.groupby('State')[numeric_cols].ffill()

In [16]:
df_updated[['Max Demand Met', 'Energy Met', 'Shortage During Peak', 'OD(+) / UD(-)', 'Energy Shortage', 'Max OD']].isnull().groupby(df_updated['State']).sum()

,Max Demand Met,Energy Met,Shortage During Peak,OD(+) / UD(-),Energy Shortage,Max OD
State,,,,,,
Andhra Pradesh,0,0,0,0,0,0
Arunachal Pradesh,0,0,0,0,0,0
Assam,0,0,0,0,0,0
Bihar,0,0,0,0,0,0
Chandigarh,0,0,0,0,0,0
Chhattisgarh,0,0,0,0,0,0
DD & DNH,0,0,0,0,0,0
DVC,0,0,0,0,0,0
Delhi,0,0,0,0,0,0


In [17]:
numeric_cols = ['Max Demand Met', 'Shortage During Peak', 'Energy Met', 
                 'Drawl Schedule', 'OD(+) / UD(-)', 'Max OD', 'Energy Shortage']

df_updated = df_updated.sort_values(['State', 'Date']).reset_index(drop=True)
df_updated[numeric_cols] = df_updated.groupby('State')[numeric_cols].bfill()

In [18]:
df_updated[['Max Demand Met', 'Energy Met', 'Shortage During Peak', 'OD(+) / UD(-)', 'Energy Shortage', 'Max OD']].isnull().groupby(df_updated['State']).sum()

,Max Demand Met,Energy Met,Shortage During Peak,OD(+) / UD(-),Energy Shortage,Max OD
State,,,,,,
Andhra Pradesh,0,0,0,0,0,0
Arunachal Pradesh,0,0,0,0,0,0
Assam,0,0,0,0,0,0
Bihar,0,0,0,0,0,0
Chandigarh,0,0,0,0,0,0
Chhattisgarh,0,0,0,0,0,0
DD & DNH,0,0,0,0,0,0
DVC,0,0,0,0,0,0
Delhi,0,0,0,0,0,0


In [19]:
df_updated.isnull().sum()

Date                    0
State                   0
Max Demand Met          0
Shortage During Peak    0
Energy Met              0
Drawl Schedule          0
OD(+) / UD(-)           0
Max OD                  0
Energy Shortage         0
dtype: int64

## Checks on the cleaned output

In [20]:
dd = df_updated[df_updated["State"] == "DD & DNH"].set_index("Date")

checks = {
    "no nulls": df_updated[numeric_cols].isna().sum().sum() == 0,
    "no Energy Met == 0": (df_updated["Energy Met"] == 0).sum() == 0,
    "no negative shortage": (df_updated[["Shortage During Peak", "Energy Shortage"]] < 0).sum().sum() == 0,
    "DD & DNH demand varies after mid-2022": dd.loc["2022-07-01":, "Max Demand Met"].nunique() > 100,
    "DD & DNH shortage ~0 after mid-2022": dd.loc["2022-07-01":, "Shortage During Peak"].mean() < 50,
    "no one-day spikes left (except real events)": int((df_updated.groupby("State")["Max Demand Met"].transform(
        lambda s: np.log(s / s.rolling(7, center=True, min_periods=4).median()).abs()) > SPIKE_LIMIT).sum())
        == len(REAL_EVENTS),
}
for name, ok in checks.items():
    print(("OK   " if ok else "FAIL ") + name)
assert all(checks.values())

national_shortage = df_updated.groupby("Date")["Shortage During Peak"].sum()
print(f"\nAverage national peak shortage: {national_shortage.mean():.0f} MW")

OK   no nulls
OK   no Energy Met == 0
OK   no negative shortage
OK   DD & DNH demand varies after mid-2022
OK   DD & DNH shortage ~0 after mid-2022
OK   no one-day spikes left (except real events)



Average national peak shortage: 412 MW


In [21]:
df_updated.to_csv('../data/processed/india_electricity_data_updated.csv', index=False)

In [22]:
time = pd.date_range(start="2021-01-01", end="2023-01-01", freq="D")
time.dayofyear

/var/folders/88/lq1yf9zd00v9hqd9g1xtfl000000gn/T/ipykernel_95522/4277899811.py:2: Pandas4Warning: DatetimeIndex.dayofyear is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_year instead.
  time.dayofyear


Index([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,
       ...
       357, 358, 359, 360, 361, 362, 363, 364, 365,   1],
      dtype='int32', length=731)

In [23]:
time

DatetimeIndex(['2021-01-01', '2021-01-02', '2021-01-03', '2021-01-04',
               '2021-01-05', '2021-01-06', '2021-01-07', '2021-01-08',
               '2021-01-09', '2021-01-10',
               ...
               '2022-12-23', '2022-12-24', '2022-12-25', '2022-12-26',
               '2022-12-27', '2022-12-28', '2022-12-29', '2022-12-30',
               '2022-12-31', '2023-01-01'],
              dtype='datetime64[us]', length=731, freq='D')

In [24]:
generator_set = pd.read_csv("../data/raw/india_generator_stack.csv")
generator_set.head()

,State,Fuel_Type,Capacity_MW,Marginal_Cost_Rs_per_MWh
0,Andhra Pradesh,Wind,4461.58,0.0
1,Andhra Pradesh,Solar,8144.32,0.0
2,Andhra Pradesh,Large Hydro,3290.00,1000.0
3,Andhra Pradesh,Small Hydro,164.51,1000.0
4,Andhra Pradesh,Coal,13890.00,2000.0
